# 필요한 라이브러리 설치

In [ ]:
!pip install -U pymupdf langgraph langchain-core langchain-upstage langchain-chroma chromadb python-dotenv

In [3]:
pip install pymupdf

# API키 설정

In [2]:
import os

os.environ["UPSTAGE_API_KEY"] = "up_A4FG7f9W6LZvC09ilmEVOBLAWgXfo"

벡터 DB 생성

In [4]:
import re
import fitz

from typing import TypedDict, List

from dotenv import load_dotenv

from langgraph.graph import StateGraph, END

from langchain_core.documents import Document
from langchain_upstage import UpstageEmbeddings
from langchain_chroma import Chroma


# =========================
# 환경 변수
# =========================

load_dotenv()

PDF_PATH = "근로기준법.pdf"

DB_DIR = "./chroma_db"

COLLECTION_NAME = "labor_law"


# =========================
# LangGraph State
# =========================

class GraphState(TypedDict):
    raw_text: str
    documents: List[Document]


# =========================
# PDF 읽기
# =========================

def load_pdf(state: GraphState):

    pdf = fitz.open(PDF_PATH)

    text = ""

    for page in pdf:
        text += page.get_text()

    return {
        **state,
        "raw_text": text
    }


# =========================
# 256자 chunk
# =========================

def make_chunks(state: GraphState):

    text = state["raw_text"]

    chunk_size = 256
    overlap = 50

    docs = []

    start = 0
    chunk_id = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():

            docs.append(
                Document(
                    page_content=chunk,
                    metadata={
                        "law": "근로기준법",
                        "chunk_id": chunk_id,
                        "start": start,
                        "end": end
                    }
                )
            )

            chunk_id += 1

        start += chunk_size - overlap

    return {
        **state,
        "documents": docs
    }


# =========================
# Vector DB 저장
# =========================

def build_vectordb(state: GraphState):

    embeddings = UpstageEmbeddings(
        model="solar-embedding-1-large-passage"
    )

    Chroma.from_documents(
        documents=state["documents"],
        embedding=embeddings,
        persist_directory=DB_DIR,
        collection_name=COLLECTION_NAME
    )

    print(f"저장 완료: {len(state['documents'])} chunks")

    return state


# =========================
# Graph 구성
# =========================

graph = StateGraph(GraphState)

graph.add_node("load_pdf", load_pdf)

graph.add_node("make_chunks", make_chunks)

graph.add_node("build_vectordb", build_vectordb)

graph.set_entry_point("load_pdf")

graph.add_edge("load_pdf", "make_chunks")

graph.add_edge("make_chunks", "build_vectordb")

graph.add_edge("build_vectordb", END)

app = graph.compile()


# =========================
# 실행
# =========================

app.invoke({
    "raw_text": "",
    "documents": []
})

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


저장 완료: 216 chunks


{'raw_text': '법제처                                                            1                                                       국가법령정보센터\n근로기준법\n \n근로기준법\n[시행 2025. 10. 23.] [법률 제20520호, 2024. 10. 22., 일부개정]\n고용노동부 (근로기준정책과 - 해고, 취업규칙, 기타) 044-202-7534\n고용노동부 (근로기준정책과 - 소년) 044-202-7535\n고용노동부 (근로기준정책과 - 임금) 044-202-7548\n고용노동부 (여성고용정책과 - 여성) 044-202-7475\n고용노동부 (임금근로시간정책과 - 근로시간, 휴게) 044-202-7545\n고용노동부 (임금근로시간정책과 - 휴일, 연차휴가) 044-202-7973\n고용노동부 (임금근로시간정책과 - 제63조 적용제외, 특례업종) 044-202-7982\n고용노동부 (임금근로시간정책과 - 유연근로시간제) 044-202-7530\n       제1장 총칙\n \n제1조(목적) 이 법은 헌법에 따라 근로조건의 기준을 정함으로써 근로자의 기본적 생활을 보장, 향상시키며 균형 있는\n국민경제의 발전을 꾀하는 것을 목적으로 한다.\n \n제2조(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2018. 3. 20., 2019. 1. 15., 2020. 5. 26.>\n1. “근로자”란 직업의 종류와 관계없이 임금을 목적으로 사업이나 사업장에 근로를 제공하는 사람을 말한다.\n2. “사용자”란 사업주 또는 사업 경영 담당자, 그 밖에 근로자에 관한 사항에 대하여 사업주를 위하여 행위하는 자를\n말한다.\n3. “근로”란 정신노동과 육체노동을 말한다.\n4. “근로계약”이란 근로자가 사용자에게 근로를 제공하고 사용자는 이에 대하여 임금을 지급하는 것을 목적으로 체\n결된 계약을 말한다.\n5. “임금”이란 사용자가 근로의 대가로 

In [5]:
from langchain_upstage import UpstageEmbeddings
from langchain_chroma import Chroma

embeddings = UpstageEmbeddings(
    model="solar-embedding-1-large-query"
)

db = Chroma(
    persist_directory="./chroma_db",
    collection_name="labor_law",
    embedding_function=embeddings
)

query = "해고 예고는 며칠 전에 해야 하나?"

results = db.similarity_search(query, k=3)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("=" * 50)

 한다.
② 정부는 제24조에 따라 해고된 근로자에 대하여 생계안정, 재취업, 직업훈련 등 필요한 조치를 우선적으로 취하여
야 한다.
 
제26조(해고의 예고) 사용자는 근로자를 해고(경영상 이유에 의한 해고를 포함한다)하려면 적어도 30일 전에 예고를 하
여야 하고, 30일 전에 예고를 하지 아니하였을 때에는 30일분 이상의 통상임금을 지급하여야 한다. 다만, 다음 각 호
의 어느 하나에 해당하는 경우에는 그러하지 아니하다. <개정 2010. 6. 4.
{'law': '근로기준법', 'chunk_id': 29, 'end': 6230, 'start': 5974}
 근로자의
과반수로 조직된 노동조합이 있는 경우에는 그 노동조합(근로자의 과반수로 조직된 노동조합이 없는 경우에는 근
로자의 과반수를 대표하는 자를 말한다. 이하 “근로자대표”라 한다)에 해고를 하려는 날의 50일 전까지 통보하고 성
실하게 협의하여야 한다.
④ 사용자는 제1항에 따라 대통령령으로 정하는 일정한 규모 이상의 인원을 해고하려면 대통령령으로 정하는 바에
따라 고용노동부장관에게 신고하여야 한다.<개정 2010. 6. 4.>
⑤ 사용자가 제1항부
{'end': 5612, 'chunk_id': 26, 'law': '근로기준법', 'start': 5356}
하는 경우
 
제27조(해고사유 등의 서면통지) ① 사용자는 근로자를 해고하려면 해고사유와 해고시기를 서면으로 통지하여야 한다.
② 근로자에 대한 해고는 제1항에 따라 서면으로 통지하여야 효력이 있다.
③ 사용자가 제26조에 따른 해고의 예고를 해고사유와 해고시기를 명시하여 서면으로 한 경우에는 제1항에 따른
통지를 한 것으로 본다.<신설 2014. 3. 24.>
 
제28조(부당해고등의 구제신청) ① 사용자가 근로자에게 부당해고등을 하면 근로자는 노동위
{'chunk_id': 31, 'start': 6386, 'end': 6642, 'law': '근로기준법'}


# QA 그래프 생성

In [16]:
from typing import TypedDict, List, Dict, Any

from langgraph.graph import StateGraph, END

# =========================
# QA State
# =========================

class QAState(TypedDict):
    question: str
    retrieved_docs: List[str]
    answer: str
    references: List[str]


# =========================
# 1. 검색 노드
# =========================

def retrieve_node(state: QAState):

    question = state["question"]

    docs = db.similarity_search(question, k=4)

    retrieved_docs = []
    references = []

    for doc in docs:

        retrieved_docs.append(doc.page_content)

        article = doc.metadata.get("article", "")

        title = doc.metadata.get("title", "")

        references.append(
            f"{article} {title}".strip()
        )

    return {
        **state,
        "retrieved_docs": retrieved_docs,
        "references": list(set(references))
    }


# =========================
# 2. 생성 노드
# =========================

def generate_node(state: QAState):

    context = "\n\n".join(
        state["retrieved_docs"]
    )

    question = state["question"]

    prompt = f"""
당신은 AI 노무사입니다.

반드시 아래 규칙을 지키세요.

- 법적 판단·확정적 의견을 내지 말 것
  ("위법입니다" 대신 "위반 가능성이 있습니다" 사용)

- 근거 조항(법률·시행령)을 가능한 한 명시할 것

- 추측·상상 금지
  데이터 없는 항목은 "정보 부족"으로 표시할 것

- 불확실하면 노동청 1350 또는
  공인노무사 상담을 권유할 것

- 2025년 기준 최저임금:
  10,030원/시간

- 한국어 존댓말 사용

[법령]
{context}

[질문]
{question}
"""

    response = llm.invoke(prompt)

    return {
        **state,
        "answer": response.content
    }


# =========================
# Graph 생성
# =========================

qa_graph_builder = StateGraph(QAState)

qa_graph_builder.add_node(
    "retrieve",
    retrieve_node
)

qa_graph_builder.add_node(
    "generate",
    generate_node
)

qa_graph_builder.set_entry_point("retrieve")

qa_graph_builder.add_edge(
    "retrieve",
    "generate"
)

qa_graph_builder.add_edge(
    "generate",
    END
)

qa_graph = qa_graph_builder.compile()

# analyze 그래프 생성

In [17]:
from typing import Optional


class AnalyzeState(TypedDict):

    contract_data: Dict[str, Any]

    rule_result: Dict[str, Any]

    user_info: Optional[Dict[str, Any]]

    summary: str

    items: List[Dict[str, Any]]


# =========================
# 분석 노드
# =========================

def analyze_node(state: AnalyzeState):

    violations = state["rule_result"].get(
        "violations",
        []
    )

    if not violations:

        return {
            **state,
            "summary": "현재 입력 정보 기준 특이사항은 발견되지 않았습니다.",
            "items": []
        }

    items = []

    for v in violations:

        prompt = f"""
당신은 AI 노무사입니다.

다음 위반 가능성을
사용자가 이해하기 쉽게 설명하세요.

반드시 아래 규칙을 지키세요.

- 법적 판단·확정적 의견 금지
  ("위법입니다" X)

- "위반 가능성이 있습니다"
  같은 표현 사용

- 근거 조항 명시

- 추측 금지

- 정보 부족 시
  "정보 부족"이라고 설명

- 불확실하면 노동청 1350
  또는 공인노무사 상담 권유

- 2025년 최저임금:
  10,030원/시간

- 한국어 존댓말 사용

제목:
{v.get("title")}

설명:
{v.get("message")}

법 조항:
{v.get("lawReference")}
"""

        response = llm.invoke(prompt)

        items.append({
            "title": v.get("title"),

            "risk_level": v.get(
                "risk",
                "medium"
            ),

            "issue": response.content,

            "law_reference": v.get(
                "lawReference",
                ""
            ),

            "action":
                "노동청 1350 또는 공인노무사 상담을 권장합니다."
        })

    summary = (
        f"{len(items)}건의 "
        "검토 항목이 발견되었습니다."
    )

    return {
        **state,
        "summary": summary,
        "items": items
    }


# =========================
# Graph 생성
# =========================

analyze_builder = StateGraph(
    AnalyzeState
)

analyze_builder.add_node(
    "analyze",
    analyze_node
)

analyze_builder.set_entry_point(
    "analyze"
)

analyze_builder.add_edge(
    "analyze",
    END
)

analyze_graph = analyze_builder.compile()

In [18]:
result = qa_graph.invoke({
    "question": "해고 예고는 며칠 전에 해야 하나?",
    "retrieved_docs": [],
    "answer": "",
    "references": []
})

print(result)

{'question': '해고 예고는 며칠 전에 해야 하나?', 'retrieved_docs': [' 한다.\n② 정부는 제24조에 따라 해고된 근로자에 대하여 생계안정, 재취업, 직업훈련 등 필요한 조치를 우선적으로 취하여\n야 한다.\n \n제26조(해고의 예고) 사용자는 근로자를 해고(경영상 이유에 의한 해고를 포함한다)하려면 적어도 30일 전에 예고를 하\n여야 하고, 30일 전에 예고를 하지 아니하였을 때에는 30일분 이상의 통상임금을 지급하여야 한다. 다만, 다음 각 호\n의 어느 하나에 해당하는 경우에는 그러하지 아니하다. <개정 2010. 6. 4.', ' 근로자의\n과반수로 조직된 노동조합이 있는 경우에는 그 노동조합(근로자의 과반수로 조직된 노동조합이 없는 경우에는 근\n로자의 과반수를 대표하는 자를 말한다. 이하 “근로자대표”라 한다)에 해고를 하려는 날의 50일 전까지 통보하고 성\n실하게 협의하여야 한다.\n④ 사용자는 제1항에 따라 대통령령으로 정하는 일정한 규모 이상의 인원을 해고하려면 대통령령으로 정하는 바에\n따라 고용노동부장관에게 신고하여야 한다.<개정 2010. 6. 4.>\n⑤ 사용자가 제1항부', '하는 경우\n \n제27조(해고사유 등의 서면통지) ① 사용자는 근로자를 해고하려면 해고사유와 해고시기를 서면으로 통지하여야 한다.\n② 근로자에 대한 해고는 제1항에 따라 서면으로 통지하여야 효력이 있다.\n③ 사용자가 제26조에 따른 해고의 예고를 해고사유와 해고시기를 명시하여 서면으로 한 경우에는 제1항에 따른\n통지를 한 것으로 본다.<신설 2014. 3. 24.>\n \n제28조(부당해고등의 구제신청) ① 사용자가 근로자에게 부당해고등을 하면 근로자는 노동위', ' 이 법에 따라 휴업한 기간과 그 후 30일 동안은 해고하지 못한다. 다만, 사용자가 제84조에 따라\n일시보상을 하였을 경우 또는 사업을 계속할 수 없게 된 경우에는 그러하지 아니하다.\n \n제24조(경영상 이유에 의한 해고의 제한) ① 사용자가 경영상 이

In [19]:
result = analyze_graph.invoke({

    "contract_data": {},

    "rule_result": {
        "violations": [
            {
                "title":
                    "최저임금 미달 가능성",

                "risk":
                    "high",

                "message":
                    "환산 시급이 최저임금보다 낮습니다.",

                "lawReference":
                    "최저임금법 제6조"
            }
        ]
    },

    "user_info": {},

    "summary": "",

    "items": []
})

print(result)

{'contract_data': {}, 'rule_result': {'violations': [{'title': '최저임금 미달 가능성', 'risk': 'high', 'message': '환산 시급이 최저임금보다 낮습니다.', 'lawReference': '최저임금법 제6조'}]}, 'user_info': {}, 'summary': '1건의 검토 항목이 발견되었습니다.', 'items': [{'title': '최저임금 미달 가능성', 'risk_level': 'high', 'issue': '**제목: 최저임금 미달 가능성**  \n\n**설명**  \n현재 계산하신 **환산 시급**이 2025년 최저임금(10,030원/시간)보다 낮은 경우, **최저임금법 제6조(최저임금의 효력)**에 따라 **위반 가능성이 있습니다**.  \n\n- **근거 조항**:  \n  > "사용자는 최저임금의 적용을 받는 근로자에게 최저임금액 이상의 임금을 지급하여야 한다." (최저임금법 제6조 1항)  \n\n- **주의 사항**:  \n  - 기본급 외 추가 수당(식대, 교통비 등)이 **최저임금 산정에 포함되는지** 여부를 반드시 확인하셔야 합니다.  \n  - 주휴수당, 연장근로수당 등 **법정 수당**이 포함된 경우, 실제 지급액 계산 방법이 달라질 수 있습니다.  \n\n- **추가 안내**:  \n  - 근로계약서, 급여명세서 등 **임금 구성 내역**이 명확하지 않을 경우, 정확한 판단이 어려울 수 있습니다.  \n  - 보다 정확한 판단을 위해 **노동청(1350)** 또는 **공인노무사**와 상담을 권장드립니다.  \n\n> ※ 본 내용은 일반적인 설명이며, 개별 사례에 따라 결과가 달라질 수 있습니다.', 'law_reference': '최저임금법 제6조', 'action': '노동청 1350 또는 공인노무사 상담을 권장합니다.'}]}
